#Objective

Same preprocessing for column values for Data preparation - 
1. Fixing the Binary values where missing is actually NO


In [14]:
import polars as pl
from pathlib import Path

In [15]:

df = pl.read_parquet(f'./data/interim/1.1-Reduced.parquet')
df.shape

(139236, 82)

In [16]:

# For columns with exactly 2 unique values where one is 9, recode 9 → 0
cols_to_recode = [
    c for c in df.columns
    if set(df[c].drop_nulls().unique().to_list()) == {9, 0}
    or (
        len(df[c].drop_nulls().unique()) == 2
        and 9 in df[c].drop_nulls().unique().to_list()
    )
]

print(f"Columns to recode (2 unique values, one is 9): {cols_to_recode}")

df = df.with_columns([
    pl.when(pl.col(c) == 9).then(0).otherwise(pl.col(c)).alias(c)
    for c in cols_to_recode
])

print("Done. Value 9 replaced with 0 in the above columns.")


Columns to recode (2 unique values, one is 9): ['Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis']
Done. Value 9 replaced with 0 in the above columns.


In [17]:
# Show all columns with at most 4 unique (non-null) values, along with those values
for c in df.columns:
    unique_vals = df[c].drop_nulls().unique().to_list()
    if len(unique_vals) <= 4:
        print(f"{c!r}: {sorted(unique_vals)}")

'Procedure': ['Knee Replacement']
'Revision Flag': [0, 1]
'Year': ['2016/17', '2017/18', '2018/19']
'Gender': [1.0, 2.0]
'Pre-Op Q Assisted': [1, 2, 9]
'Pre-Op Q Assisted By': [0]
'Pre-Op Q Previous Surgery': [1, 2, 9]
'Pre-Op Q Disability': [1, 2, 9]
'Heart Disease': [0, 1]
'High Bp': [0, 1]
'Stroke': [0, 1]
'Circulation': [0, 1]
'Lung Disease': [0, 1]
'Diabetes': [0, 1]
'Kidney Disease': [0, 1]
'Nervous System': [0, 1]
'Liver Disease': [0, 1]
'Cancer': [0, 1]
'Depression': [0, 1]
'Arthritis': [0, 1]
'Pre-Op Q Mobility': [1, 2, 3, 9]
'Pre-Op Q Self-Care': [1, 2, 3, 9]
'Pre-Op Q Activity': [1, 2, 3, 9]
'Pre-Op Q Discomfort': [1, 2, 3, 9]
'Pre-Op Q Anxiety': [1, 2, 3, 9]
'Post-Op Q Assisted': [1, 2, 9]
'Post-Op Q Disability': [1, 2, 9]
'Post-Op Q Mobility': [1, 2, 3, 9]
'Post-Op Q Self-Care': [1, 2, 3, 9]
'Post-Op Q Activity': [1, 2, 3, 9]
'Post-Op Q Discomfort': [1, 2, 3, 9]
'Post-Op Q Anxiety': [1, 2, 3, 9]
'Post-Op Q Allergy': [1, 2, 9]
'Post-Op Q Bleeding': [1, 2, 9]
'Post-Op Q Woun

In [18]:
# Recode 'Pre-Op Q Assisted By': 9 → 0, anything else → 1
df = df.with_columns(
    pl.when(pl.col("Pre-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Pre-Op Q Assisted By")
)

print("'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Pre-Op Q Assisted By"].value_counts())

#Post-Op Q Assisted By 

df = df.with_columns(
    pl.when(pl.col("Post-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Post-Op Q Assisted By")
)

print("'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Post-Op Q Assisted By"].value_counts())


'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (1, 2)
┌──────────────────────┬────────┐
│ Pre-Op Q Assisted By ┆ count  │
│ ---                  ┆ ---    │
│ i32                  ┆ u32    │
╞══════════════════════╪════════╡
│ 1                    ┆ 139236 │
└──────────────────────┴────────┘
'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (2, 2)
┌───────────────────────┬────────┐
│ Post-Op Q Assisted By ┆ count  │
│ ---                   ┆ ---    │
│ i32                   ┆ u32    │
╞═══════════════════════╪════════╡
│ 1                     ┆ 10864  │
│ 0                     ┆ 128372 │
└───────────────────────┴────────┘


In [19]:
# Replace common missing value indicators with null
# Numeric and string columns must be handled separately — is_in() requires matching types
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]
numeric_replace = [9, 999]
string_replace  = ["*", ""]

numeric_exprs = [
    pl.when(pl.col(c).is_in(numeric_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype in numeric_dtypes
]
string_exprs = [
    pl.when(pl.col(c).is_in(string_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype == pl.Utf8
]

df = df.with_columns(numeric_exprs + string_exprs)
print(df.null_count())

shape: (1, 82)
┌─────────────┬───────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬─────────┐
│ Provider    ┆ Procedure ┆ Revision   ┆ Year ┆ … ┆ Knee Repla ┆ Knee Repla ┆ Knee Repla ┆ CSVYear │
│ Code        ┆ ---       ┆ Flag       ┆ ---  ┆   ┆ cement     ┆ cement     ┆ cement OKS ┆ ---     │
│ ---         ┆ u32       ┆ ---        ┆ u32  ┆   ┆ Post-Op Q  ┆ Post-Op Q  ┆ Post-Op Q… ┆ u32     │
│ u32         ┆           ┆ u32        ┆      ┆   ┆ Sta…       ┆ Sco…       ┆ ---        ┆         │
│             ┆           ┆            ┆      ┆   ┆ ---        ┆ ---        ┆ u32        ┆         │
│             ┆           ┆            ┆      ┆   ┆ u32        ┆ u32        ┆            ┆         │
╞═════════════╪═══════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪═════════╡
│ 0           ┆ 0         ┆ 0          ┆ 0    ┆ … ┆ 980        ┆ 2960       ┆ 5381       ┆ 0       │
└─────────────┴───────────┴────────────┴──────┴───┴────────────┴────────────

In [20]:

# Column type summary: numeric vs categorical
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]

numeric_cols = [c for c in df.columns if df[c].dtype in numeric_dtypes]
cat_cols     = [c for c in df.columns if df[c].dtype in (pl.Utf8, pl.Categorical)]
other_cols   = [c for c in df.columns if c not in numeric_cols and c not in cat_cols]

print(f"Numeric columns     : {len(numeric_cols)}")
print(f"Categorical columns : {len(cat_cols)}")
print(f"  Names: {cat_cols}")
if other_cols:
    print(f"Other columns       : {len(other_cols)} — {[(c, df[c].dtype) for c in other_cols]}")


Numeric columns     : 77
Categorical columns : 5
  Names: ['Provider Code', 'Procedure', 'Year', 'Age Band', 'CSVYear']


In [21]:

# Unique values for each categorical column (to plan encoding/transforms)
for c in cat_cols:
    unique_vals = df[c].drop_nulls().unique().cast(pl.Utf8).sort().to_list()
    print(f"\n{c!r}  (dtype={df[c].dtype}, {len(unique_vals)} unique values)")
    print(f"  {unique_vals}")



'Provider Code'  (dtype=Categorical, 294 unique values)
  ['ADP02', 'AHH', 'AVQ', 'NFH01', 'NN401', 'NN801', 'NQM01', 'NT202', 'NT204', 'NT205', 'NT206', 'NT209', 'NT210', 'NT211', 'NT212', 'NT213', 'NT214', 'NT215', 'NT218', 'NT219', 'NT224', 'NT225', 'NT226', 'NT229', 'NT230', 'NT233', 'NT235', 'NT237', 'NT238', 'NT239', 'NT241', 'NT242', 'NT244', 'NT245', 'NT301', 'NT302', 'NT304', 'NT305', 'NT308', 'NT309', 'NT30A', 'NT310', 'NT312', 'NT313', 'NT314', 'NT315', 'NT316', 'NT317', 'NT318', 'NT319', 'NT320', 'NT321', 'NT322', 'NT324', 'NT325', 'NT327', 'NT332', 'NT333', 'NT337', 'NT339', 'NT343', 'NT344', 'NT345', 'NT347', 'NT348', 'NT350', 'NT351', 'NT364', 'NT401', 'NT402', 'NT403', 'NT404', 'NT405', 'NT406', 'NT408', 'NT409', 'NT410', 'NT411', 'NT412', 'NT413', 'NT414', 'NT416', 'NT417', 'NT418', 'NT419', 'NT420', 'NT421', 'NT422', 'NT423', 'NT424', 'NT427', 'NT428', 'NT429', 'NT430', 'NT431', 'NT432', 'NT433', 'NT434', 'NT435', 'NT436', 'NT437', 'NT438', 'NT439', 'NT440', 'NT441',

In [22]:

# Label-encode selected categorical columns as integers, starting from 1 (0 is reserved for "No")
# Ordered: Year, CSVYear (chronological), Age Band (natural age order)
# Unordered: Procedure (alphabetical)

cols_to_encode = ["Procedure", "Year", "Age Band", "Provider Code"   ]  # --- IGNORE ---

age_band_order = sorted(df["Age Band"].drop_nulls().unique().cast(pl.Utf8).to_list())
# ↑ Replace with explicit list if alphabetical sort does not match the natural age order
# e.g. age_band_order = ["Under 45", "45 to 49", "50 to 54", "55 to 59", "60 to 64",
#                        "65 to 69", "70 to 74", "75 to 79", "80 to 84", "85 to 89", "90 and over"]

explicit_orders = {
    "Year":     sorted(df["Year"].drop_nulls().unique().cast(pl.Utf8).to_list()),
    "CSVYear":  sorted(df["CSVYear"].drop_nulls().unique().cast(pl.Utf8).to_list()),
    "Age Band": age_band_order,
}

encoding_maps = {}
for c in cols_to_encode:
    order = explicit_orders.get(c, sorted(df[c].drop_nulls().unique().cast(pl.Utf8).to_list()))
    encoding_maps[c] = {label: code for code, label in enumerate(order, start=1)}

# Apply encoding in-place (categorical → Int32)
df = df.with_columns([
    pl.col(c)
      .cast(pl.Utf8)
      .replace({k: str(v) for k, v in encoding_maps[c].items()})
      .cast(pl.Int32)
      .alias(c)
    for c in cols_to_encode
])

# Print mappings for reference
for c, mapping in encoding_maps.items():
    print(f"\n{c!r}")
    for label, code in mapping.items():
        print(f"  {code:>3} → {label!r}")



'Procedure'
    1 → 'Knee Replacement'

'Year'
    1 → '2016/17'
    2 → '2017/18'
    3 → '2018/19'

'Age Band'
    1 → '40 to 49'
    2 → '50 to 59'
    3 → '60 to 69'
    4 → '70 to 79'
    5 → '80 to 89'
    6 → '90 to 120'

'Provider Code'
    1 → 'ADP02'
    2 → 'AHH'
    3 → 'AVQ'
    4 → 'NFH01'
    5 → 'NN401'
    6 → 'NN801'
    7 → 'NQM01'
    8 → 'NT202'
    9 → 'NT204'
   10 → 'NT205'
   11 → 'NT206'
   12 → 'NT209'
   13 → 'NT210'
   14 → 'NT211'
   15 → 'NT212'
   16 → 'NT213'
   17 → 'NT214'
   18 → 'NT215'
   19 → 'NT218'
   20 → 'NT219'
   21 → 'NT224'
   22 → 'NT225'
   23 → 'NT226'
   24 → 'NT229'
   25 → 'NT230'
   26 → 'NT233'
   27 → 'NT235'
   28 → 'NT237'
   29 → 'NT238'
   30 → 'NT239'
   31 → 'NT241'
   32 → 'NT242'
   33 → 'NT244'
   34 → 'NT245'
   35 → 'NT301'
   36 → 'NT302'
   37 → 'NT304'
   38 → 'NT305'
   39 → 'NT308'
   40 → 'NT309'
   41 → 'NT30A'
   42 → 'NT310'
   43 → 'NT312'
   44 → 'NT313'
   45 → 'NT314'
   46 → 'NT315'
   47 → 'NT316'
   48 

In [23]:

# Drop columns that are not useful for modelling
cols_to_drop = [
                           # unique per provider, not a useful feature
    "Knee Replacement EQ 5D Index Post-Op Q Predicted",
    "Knee Replacement EQ VAS_Post-Op Q Predicted",
     "Knee Replacement OKS Post-Op Q Predicted",
    "CSVYear",                                # redundant with Year after encoding
]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"Remaining columns: {df.shape[1]}")


Dropped 4 columns: ['Knee Replacement EQ 5D Index Post-Op Q Predicted', 'Knee Replacement EQ VAS_Post-Op Q Predicted', 'Knee Replacement OKS Post-Op Q Predicted', 'CSVYear']
Remaining columns: 78


In [24]:
# Identify columns to remove and keep
all_columns = df.columns  # Polars returns a plain list

# Columns to keep even if they contain 'Post-Op'
keep_post_op = ['Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Post-Op Q EQ VAS', 'Hip Replacement Post-Op Q Score']

# Remove Post-Op columns (except the specified ones), Predicted columns, and CSVYear
removed_columns = [
    col for col in all_columns
    if ('Post-Op' in col and col not in keep_post_op) or 'Predicted' in col or col == 'CSVYear'
]
kept_columns = [col for col in all_columns if col not in removed_columns]

print("Columns to be removed:")
print(removed_columns)
print(f"\nTotal removed: {len(removed_columns)}")

print("\nColumns to be kept:")
print(kept_columns)
print(f"\nTotal kept: {len(kept_columns)}")

Columns to be removed:
['Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Knee Replacement Post-Op Q Pain', 'Knee Replacement Post-Op Q Night Pain', 'Knee Replacement Post-Op Q Washing', 'Knee Replacement Post-Op Q Transport', 'Knee Replacement Post-Op Q Walking', 'Knee Replacement Post-Op Q Standing', 'Knee Replacement Post-Op Q Limping', 'Knee Replacement Post-Op Q Kneeling', 'Knee Replacement Post-Op Q Work', 'Knee Replacement Post-Op Q Confidence', 'Knee Replacement Post-Op Q Shopping', 'Knee Replacement Post-Op Q Stairs', 'Knee Replacement Post-Op Q Score']

Total removed: 30

Columns to be kept:
['Provider Code', 'Procedure', 'Revision Fla

In [25]:

df.write_parquet(f'./data/interim/2.0-preprocessing.parquet', compression='gzip')